# 05 — Pipeline Validation

**Project:** Orchestrated Incremental Orders Pipeline with Databricks Workflows  
**Layer:** Validation  
**Source Tables:**

- `orders_workflow_project.bronze_orders_raw`
- `orders_workflow_project.silver_orders_current`
- `orders_workflow_project.gold_orders_by_status`
- `orders_workflow_project.gold_daily_sales`
- `orders_workflow_project.gold_order_summary`
- `orders_workflow_project.pipeline_run_audit`

**Target Table:**

- `orders_workflow_project.pipeline_validation_results`

This notebook validates the final state of the orchestrated pipeline.

It checks that:

- All expected tables exist.
- Bronze contains 10 raw events.
- Silver contains 5 current orders.
- Gold tables contain the expected number of records.
- Silver contains the expected latest status per order.
- Audit records were created.
- The pipeline can be considered successful.

In [0]:
dbutils.widgets.text("pipeline_run_id", "manual_run", "Pipeline Run ID")

pipeline_run_id = dbutils.widgets.get("pipeline_run_id")

print(f"Pipeline Run ID: {pipeline_run_id}")

In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, LongType, TimestampType
)
from pyspark.sql.functions import current_timestamp

In [0]:
spark.sql("USE SCHEMA orders_workflow_project")

In [0]:
schema_name = "orders_workflow_project"

expected_tables = {
    "source_orders_batch_1": 3,
    "source_orders_batch_2": 3,
    "source_orders_batch_3": 4,
    "bronze_orders_raw": 10,
    "silver_orders_current": 5,
    "gold_orders_by_status": 4,
    "gold_daily_sales": 2,
    "gold_order_summary": 1
}

validation_table = f"{schema_name}.pipeline_validation_results"
audit_table = f"{schema_name}.pipeline_run_audit"

print("Expected tables:")
for table_name, expected_count in expected_tables.items():
    print(f"- {table_name}: {expected_count}")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {validation_table} (
    pipeline_run_id STRING,
    validation_name STRING,
    expected_value STRING,
    actual_value STRING,
    status STRING,
    message STRING,
    validated_ts TIMESTAMP
)
USING DELTA
""")

print(f"Validation table ready: {validation_table}")

In [0]:
tables_df = spark.sql(f"SHOW TABLES IN {schema_name}")

existing_tables = [
    row["tableName"]
    for row in tables_df.collect()
]

print("Existing tables:")
for table_name in existing_tables:
    print(f"- {table_name}")

In [0]:
validation_results = []

for table_name in expected_tables.keys():
    exists = table_name in existing_tables

    validation_results.append({
        "pipeline_run_id": pipeline_run_id,
        "validation_name": f"table_exists_{table_name}",
        "expected_value": "EXISTS",
        "actual_value": "EXISTS" if exists else "MISSING",
        "status": "PASSED" if exists else "FAILED",
        "message": f"Table {table_name} exists." if exists else f"Table {table_name} is missing."
    })

display(
    spark.createDataFrame(validation_results)
)

In [0]:
for table_name, expected_count in expected_tables.items():
    full_table_name = f"{schema_name}.{table_name}"

    if table_name in existing_tables:
        actual_count = spark.table(full_table_name).count()

        validation_results.append({
            "pipeline_run_id": pipeline_run_id,
            "validation_name": f"record_count_{table_name}",
            "expected_value": str(expected_count),
            "actual_value": str(actual_count),
            "status": "PASSED" if actual_count == expected_count else "FAILED",
            "message": (
                f"Table {table_name} has expected count {expected_count}."
                if actual_count == expected_count
                else f"Table {table_name} expected {expected_count}, got {actual_count}."
            )
        })
    else:
        validation_results.append({
            "pipeline_run_id": pipeline_run_id,
            "validation_name": f"record_count_{table_name}",
            "expected_value": str(expected_count),
            "actual_value": "TABLE_MISSING",
            "status": "FAILED",
            "message": f"Cannot validate count because table {table_name} is missing."
        })

display(
    spark.createDataFrame(validation_results)
)

In [0]:
expected_silver_status = {
    "1001": "delivered",
    "1002": "cancelled",
    "1003": "shipped",
    "1004": "delivered",
    "1005": "pending"
}

In [0]:
silver_status_df = spark.sql("""
SELECT
    order_id,
    order_status
FROM orders_workflow_project.silver_orders_current
ORDER BY order_id
""")

display(silver_status_df)

actual_silver_status = {
    row["order_id"]: row["order_status"]
    for row in silver_status_df.collect()
}

In [0]:
for order_id, expected_status in expected_silver_status.items():
    actual_status = actual_silver_status.get(order_id)

    validation_results.append({
        "pipeline_run_id": pipeline_run_id,
        "validation_name": f"silver_status_order_{order_id}",
        "expected_value": expected_status,
        "actual_value": actual_status if actual_status is not None else "ORDER_MISSING",
        "status": "PASSED" if actual_status == expected_status else "FAILED",
        "message": (
            f"Order {order_id} has expected status {expected_status}."
            if actual_status == expected_status
            else f"Order {order_id} expected {expected_status}, got {actual_status}."
        )
    })

display(
    spark.createDataFrame(validation_results)
)

In [0]:
audit_exists = "pipeline_run_audit" in existing_tables

if audit_exists:
    audit_count = spark.table(audit_table).count()

    validation_results.append({
        "pipeline_run_id": pipeline_run_id,
        "validation_name": "audit_records_exist",
        "expected_value": ">= 1",
        "actual_value": str(audit_count),
        "status": "PASSED" if audit_count >= 1 else "FAILED",
        "message": (
            f"Audit table contains {audit_count} records."
            if audit_count >= 1
            else "Audit table exists but contains no records."
        )
    })
else:
    validation_results.append({
        "pipeline_run_id": pipeline_run_id,
        "validation_name": "audit_records_exist",
        "expected_value": ">= 1",
        "actual_value": "TABLE_MISSING",
        "status": "FAILED",
        "message": "Audit table is missing."
    })

display(
    spark.createDataFrame(validation_results)
)

In [0]:
expected_audit_tasks = [
    "01_create_sample_batches",
    "02_bronze_ingestion_batch_1",
    "02_bronze_ingestion_batch_2",
    "02_bronze_ingestion_batch_3",
    "03_silver_merge_upsert_batch_1",
    "03_silver_merge_upsert_batch_2",
    "03_silver_merge_upsert_batch_3",
    "04_gold_business_metrics"
]

In [0]:
if audit_exists:
    audit_tasks_df = spark.sql(f"""
    SELECT DISTINCT task_name
    FROM {audit_table}
    WHERE pipeline_run_id = '{pipeline_run_id.replace("'", "''")}'
    """)

    audit_tasks = [
        row["task_name"]
        for row in audit_tasks_df.collect()
    ]

    print("Audit tasks found:")
    for task in audit_tasks:
        print(f"- {task}")

    for expected_task in expected_audit_tasks:
        task_found = expected_task in audit_tasks

        validation_results.append({
            "pipeline_run_id": pipeline_run_id,
            "validation_name": f"audit_task_{expected_task}",
            "expected_value": "FOUND",
            "actual_value": "FOUND" if task_found else "MISSING",
            "status": "PASSED" if task_found else "FAILED",
            "message": (
                f"Audit task {expected_task} was found."
                if task_found
                else f"Audit task {expected_task} is missing for pipeline_run_id {pipeline_run_id}."
            )
        })
else:
    for expected_task in expected_audit_tasks:
        validation_results.append({
            "pipeline_run_id": pipeline_run_id,
            "validation_name": f"audit_task_{expected_task}",
            "expected_value": "FOUND",
            "actual_value": "AUDIT_TABLE_MISSING",
            "status": "FAILED",
            "message": f"Cannot validate audit task {expected_task} because audit table is missing."
        })

display(
    spark.createDataFrame(validation_results)
)

In [0]:
validation_schema = StructType([
    StructField("pipeline_run_id", StringType(), False),
    StructField("validation_name", StringType(), False),
    StructField("expected_value", StringType(), False),
    StructField("actual_value", StringType(), False),
    StructField("status", StringType(), False),
    StructField("message", StringType(), False)
])

validation_results_df = spark.createDataFrame(validation_results, validation_schema)

validation_results_df = validation_results_df.withColumn(
    "validated_ts",
    current_timestamp()
)

display(validation_results_df)

In [0]:
pipeline_run_id_sql = pipeline_run_id.replace("'", "''")

spark.sql(f"""
DELETE FROM {validation_table}
WHERE pipeline_run_id = '{pipeline_run_id_sql}'
""")

print(f"Previous validation results deleted for pipeline_run_id: {pipeline_run_id}")

In [0]:
validation_results_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(validation_table)

print("Validation results saved successfully.")

In [0]:
validation_summary_df = spark.sql(f"""
SELECT
    status,
    COUNT(*) AS total_validations
FROM {validation_table}
WHERE pipeline_run_id = '{pipeline_run_id_sql}'
GROUP BY status
ORDER BY status
""")

display(validation_summary_df)

In [0]:
failed_validations_df = spark.sql(f"""
SELECT
    validation_name,
    expected_value,
    actual_value,
    message
FROM {validation_table}
WHERE pipeline_run_id = '{pipeline_run_id_sql}'
  AND status = 'FAILED'
ORDER BY validation_name
""")

display(failed_validations_df)

In [0]:
failed_count = failed_validations_df.count()
total_validations = validation_results_df.count()

if failed_count == 0:
    final_status = "SUCCESS"
    final_message = f"Pipeline validation completed successfully. {total_validations} validations passed."
else:
    final_status = "FAILED"
    final_message = f"Pipeline validation failed. {failed_count} validations failed out of {total_validations}."

print(f"Final validation status: {final_status}")
print(final_message)

In [0]:
final_message_sql = final_message.replace("'", "''")

spark.sql(f"""
INSERT INTO {audit_table}
SELECT
    '{pipeline_run_id_sql}' AS pipeline_run_id,
    '05_pipeline_validation' AS task_name,
    '{final_status}' AS status,
    CAST({int(total_validations)} AS BIGINT) AS records_processed,
    '{final_message_sql}' AS message,
    current_timestamp() AS processed_ts
""")

print("Validation audit record inserted successfully.")

In [0]:
if failed_count > 0:
    raise Exception(final_message)

print("Pipeline validation passed successfully.")

In [0]:
display(
    spark.sql(f"""
    SELECT *
    FROM {validation_table}
    WHERE pipeline_run_id = '{pipeline_run_id_sql}'
    ORDER BY validation_name
    """)
)

In [0]:
display(
    spark.sql(f"""
    SELECT *
    FROM {audit_table}
    WHERE pipeline_run_id = '{pipeline_run_id_sql}'
    ORDER BY processed_ts
    """)
)